# 从零实现 U-Net：像素级分割、奇偶尺寸对齐与工程评估

这份 notebook 只使用 PyTorch 基础层，手写 `DoubleConv`、`Down`、`Up` 和 `UNet` 的 `forward`，完整走通二值语义分割：像素 logits、BCEWithLogits + Dice、空 mask 指标策略、奇数尺寸 skip 对齐、合成几何图形小样本过拟合、validation 阈值冻结、重叠 tile 推理和权重指纹。

模型在 CPU 上训练极小合成数据；它用于验证实现和评估合同，不代表医学、遥感、文档或自动驾驶分割效果。


## 1. 分割系统的 shape 主线

```text
image [N,C,H,W]
  -> 编码器：DoubleConv + Down，增加语义、降低分辨率
  -> bottleneck
  -> 解码器：上采样 + 对齐 skip + concat + DoubleConv
  -> pixel logits [N,1,H,W]
  -> sigmoid 仅用于概率/阈值
  -> validation 选择阈值 -> 冻结 -> test
```

二值分割最后一层输出一个通道；多类互斥分割通常输出 `C` 通道并配合 cross-entropy。不要把类别分类的 `[N,C]` logits 和像素级 `[N,C,H,W]` logits 混为一谈。


In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from hashlib import sha256  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 230728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。


## 2. 为什么奇数尺寸会让 skip connection 对不上

`MaxPool2d(2)` 对奇数长度执行向下取整，而 `ConvTranspose2d(..., stride=2)` 通常精确翻倍。例如 31 经池化变 15，再上采样回 30，不会自动回到 31。U-Net 要把解码特征与同层编码特征在通道维拼接，空间维必须先对齐。

下面的 `align_to` 采用中心裁剪/对称 padding：source 太大先裁，太小再补。生产必须固定 padding mode 和左右分配规则，否则训练、导出与服务端可能相差一像素。


In [ ]:
def align_to(source, reference):  # 定义本节可复用的核心函数。
    if source.ndim != 4 or reference.ndim != 4:  # 按当前条件选择后续控制路径。
        raise ValueError("align_to 只接受 NCHW 张量")  # 遇到非法合同立即显式失败。
    target_h, target_w = reference.shape[-2:]  # 计算并保存当前步骤的中间状态。
    height, width = source.shape[-2:]  # 计算并保存当前步骤的中间状态。

    if height > target_h:  # 按当前条件选择后续控制路径。
        top = (height - target_h) // 2  # 计算并保存当前步骤的中间状态。
        source = source[..., top:top + target_h, :]  # 计算并保存当前步骤的中间状态。
    if width > target_w:  # 按当前条件选择后续控制路径。
        left = (width - target_w) // 2  # 计算并保存当前步骤的中间状态。
        source = source[..., :, left:left + target_w]  # 计算并保存当前步骤的中间状态。

    pad_h = target_h - source.shape[-2]  # 计算并保存当前步骤的中间状态。
    pad_w = target_w - source.shape[-1]  # 计算并保存当前步骤的中间状态。
    if pad_h < 0 or pad_w < 0:  # 按当前条件选择后续控制路径。
        raise RuntimeError("裁剪后仍大于目标尺寸")  # 遇到非法合同立即显式失败。
    source = F.pad(source, [pad_w // 2, pad_w - pad_w // 2,  # 计算并保存当前步骤的中间状态。
                            pad_h // 2, pad_h - pad_h // 2])  # 执行当前语句以推进本节示例。
    return source  # 返回当前分支计算出的结果。

small = torch.ones(1, 2, 4, 6)  # 计算并保存当前步骤的中间状态。
large = torch.ones(1, 2, 7, 9)  # 计算并保存当前步骤的中间状态。
assert align_to(small, large).shape == large.shape  # 用受控断言验证关键不变量。
assert align_to(large, small).shape == small.shape  # 用受控断言验证关键不变量。
assert float(align_to(small, large).sum()) == float(small.sum())  # 用受控断言验证关键不变量。


## 3. 编码器：DoubleConv 与 Down

原始 U-Net 每一级使用两个 3×3 VALID 卷积，所以 skip 需要中心裁剪；很多现代实现改用 `padding=1` 的 SAME 卷积。这里采用现代 SAME 变体，但 pooling 遇到奇数尺寸仍需显式对齐。

`DoubleConv` 不改变空间尺寸，`Down` 先做 2×2 最大池化再提取特征。卷积后不用 sigmoid，让中间表示保留无界数值范围。


In [ ]:
class DoubleConv(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
            nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),  # 计算并保存当前步骤的中间状态。
            nn.ReLU(inplace=False),  # 计算并保存当前步骤的中间状态。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.net(x)  # 返回当前分支计算出的结果。

class Down(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels, out_channels):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.net(x)  # 返回当前分支计算出的结果。

encoder_probe = Down(4, 8)(torch.zeros(2, 4, 31, 35))  # 计算并保存当前步骤的中间状态。
assert encoder_probe.shape == (2, 8, 15, 17)  # 用受控断言验证关键不变量。
assert sum(isinstance(m, nn.Conv2d) for m in DoubleConv(1, 4).modules()) == 2  # 用受控断言验证关键不变量。


## 4. 解码器：上采样、skip concat 与逐像素 logits

`Up` 先用转置卷积扩大 decoder feature，再对齐到 encoder skip 的空间尺寸，沿 channel 维 `dim=1` 拼接，最后用 `DoubleConv` 融合。拼接保留编码器的高分辨率定位信息；相加则要求通道一致且会混合两路语义，不是同一操作。

最后的 1×1 卷积把每个像素的 feature 映射为一个 logit。`forward` 不做 sigmoid，因为 `BCEWithLogitsLoss` 将 sigmoid 与交叉熵组合，数值更稳定。


In [ ]:
class Up(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, decoder_channels, skip_channels, out_channels):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.up = nn.ConvTranspose2d(decoder_channels, out_channels, kernel_size=2, stride=2)  # 计算并保存当前步骤的中间状态。
        self.fuse = DoubleConv(out_channels + skip_channels, out_channels)  # 计算并保存当前步骤的中间状态。

    def forward(self, decoder_feature, skip_feature):  # 定义本节可复用的核心函数。
        decoder_feature = align_to(self.up(decoder_feature), skip_feature)  # 计算并保存当前步骤的中间状态。
        merged = torch.cat([skip_feature, decoder_feature], dim=1)  # 计算并保存当前步骤的中间状态。
        return self.fuse(merged)  # 返回当前分支计算出的结果。

class UNet(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_channels=1, out_channels=1, base=4):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.inc = DoubleConv(in_channels, base)  # 计算并保存当前步骤的中间状态。
        self.down1 = Down(base, base * 2)  # 计算并保存当前步骤的中间状态。
        self.down2 = Down(base * 2, base * 4)  # 计算并保存当前步骤的中间状态。
        self.up1 = Up(base * 4, base * 2, base * 2)  # 计算并保存当前步骤的中间状态。
        self.up2 = Up(base * 2, base, base)  # 计算并保存当前步骤的中间状态。
        self.out_conv = nn.Conv2d(base, out_channels, kernel_size=1)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        if x.ndim != 4:  # 按当前条件选择后续控制路径。
            raise ValueError("UNet 期望 [N,C,H,W]")  # 遇到非法合同立即显式失败。
        x1 = self.inc(x)  # 计算并保存当前步骤的中间状态。
        x2 = self.down1(x1)  # 计算并保存当前步骤的中间状态。
        bottleneck = self.down2(x2)  # 计算并保存当前步骤的中间状态。
        decoded = self.up1(bottleneck, x2)  # 计算并保存当前步骤的中间状态。
        decoded = self.up2(decoded, x1)  # 计算并保存当前步骤的中间状态。
        logits = self.out_conv(decoded)  # 计算并保存当前步骤的中间状态。
        if logits.shape[-2:] != x.shape[-2:]:  # 按当前条件选择后续控制路径。
            raise RuntimeError("输出与输入空间尺寸不一致")  # 遇到非法合同立即显式失败。
        return logits  # 返回当前分支计算出的结果。

unet = UNet().to(DEVICE)  # 计算并保存当前步骤的中间状态。
odd_logits = unet(torch.zeros(2, 1, 31, 35))  # 计算并保存当前步骤的中间状态。
assert odd_logits.shape == (2, 1, 31, 35)  # 用受控断言验证关键不变量。
assert odd_logits.dtype.is_floating_point  # 用受控断言验证关键不变量。
assert odd_logits.requires_grad  # 用受控断言验证关键不变量。
assert torch.isfinite(odd_logits).all()  # 用受控断言验证关键不变量。
assert not any(isinstance(module, nn.Sigmoid) for module in unet.modules())  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in unet.parameters()) < 20_000  # 用受控断言验证关键不变量。


## 5. BCEWithLogits + soft Dice

像素 BCE 对每个位置提供稳定梯度，但前景极少时容易被大量背景主导。soft Dice 直接优化区域重叠：

$$\mathrm{Dice}=\frac{2\sum_i p_i y_i+\epsilon}{\sum_i p_i+\sum_i y_i+\epsilon}.$$

组合损失 $L=\alpha L_{BCE}+(1-\alpha)(1-\mathrm{Dice})$ 同时保留像素概率项与区域项。这里逐样本计算再求均值，避免大目标完全压过小目标；`eps` 还定义了空预测/空真值时的数值行为。


In [ ]:
def soft_dice_loss(logits, targets, eps=1e-6):  # 定义本节可复用的核心函数。
    if logits.shape != targets.shape:  # 按当前条件选择后续控制路径。
        raise ValueError(f"logits/targets shape 不同: {logits.shape} vs {targets.shape}")  # 遇到非法合同立即显式失败。
    probabilities = torch.sigmoid(logits)  # 计算并保存当前步骤的中间状态。
    dims = tuple(range(1, logits.ndim))  # 计算并保存当前步骤的中间状态。
    intersection = (probabilities * targets).sum(dim=dims)  # 计算并保存当前步骤的中间状态。
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)  # 计算并保存当前步骤的中间状态。
    dice = (2 * intersection + eps) / (denominator + eps)  # 计算并保存当前步骤的中间状态。
    return 1 - dice.mean()  # 返回当前分支计算出的结果。

def segmentation_loss(logits, targets, bce_weight=0.5):  # 定义本节可复用的核心函数。
    bce = F.binary_cross_entropy_with_logits(logits, targets)  # 计算并保存当前步骤的中间状态。
    dice = soft_dice_loss(logits, targets)  # 计算并保存当前步骤的中间状态。
    return bce_weight * bce + (1 - bce_weight) * dice  # 返回当前分支计算出的结果。

perfect_logits = torch.tensor([[[[12.0, -12.0], [-12.0, 12.0]]]])  # 计算并保存当前步骤的中间状态。
perfect_target = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])  # 计算并保存当前步骤的中间状态。
wrong_logits = -perfect_logits  # 计算并保存当前步骤的中间状态。
assert segmentation_loss(perfect_logits, perfect_target) < 0.01  # 用受控断言验证关键不变量。
assert segmentation_loss(wrong_logits, perfect_target) > 5.0  # 用受控断言验证关键不变量。
assert torch.isfinite(segmentation_loss(torch.zeros_like(perfect_logits), perfect_target))  # 用受控断言验证关键不变量。


## 6. IoU/Dice 与空 mask 必须先约定

硬预测下 $IoU=|P\cap G|/|P\cup G|$，$Dice=2|P\cap G|/(|P|+|G|)$。当预测和真值都为空，分母为零：

- `perfect`：记 1，适合“正确判空也有价值”的任务；
- `skip`：该样本不进入重叠均值，但应另报空样本识别率；
- 若真值空但出现假阳性，IoU/Dice 仍为 0，不能跳过。

报告必须写明 policy、逐样本还是全局聚合、阈值和 resize 后处理。


In [ ]:
def binary_overlap_metrics(probabilities, targets, threshold=0.5, empty_policy="perfect"):  # 定义本节可复用的核心函数。
    if probabilities.shape != targets.shape:  # 按当前条件选择后续控制路径。
        raise ValueError("probabilities/targets shape 不一致")  # 遇到非法合同立即显式失败。
    if empty_policy not in {"perfect", "skip"}:  # 按当前条件选择后续控制路径。
        raise ValueError("empty_policy 必须是 perfect 或 skip")  # 遇到非法合同立即显式失败。
    predictions = probabilities >= threshold  # 计算并保存当前步骤的中间状态。
    truth = targets >= 0.5  # 计算并保存当前步骤的中间状态。
    ious, dices, empty_correct = [], [], 0  # 计算并保存当前步骤的中间状态。
    for prediction, target in zip(predictions, truth):  # 遍历输入元素以累积或检查结果。
        intersection = int((prediction & target).sum())  # 计算并保存当前步骤的中间状态。
        union = int((prediction | target).sum())  # 计算并保存当前步骤的中间状态。
        total = int(prediction.sum() + target.sum())  # 计算并保存当前步骤的中间状态。
        both_empty = union == 0  # 计算并保存当前步骤的中间状态。
        if both_empty:  # 按当前条件选择后续控制路径。
            empty_correct += 1  # 计算并保存当前步骤的中间状态。
            if empty_policy == "skip":  # 按当前条件选择后续控制路径。
                continue  # 调整当前循环或占位控制流。
        ious.append(1.0 if both_empty else intersection / union)  # 执行当前语句以推进本节示例。
        dices.append(1.0 if both_empty else 2 * intersection / total)  # 执行当前语句以推进本节示例。
    return {"iou": float(np.mean(ious)) if ious else math.nan,  # 返回当前分支计算出的结果。
            "dice": float(np.mean(dices)) if dices else math.nan,  # 执行当前语句以推进本节示例。
            "evaluated": len(ious), "empty_correct": empty_correct}  # 执行当前语句以推进本节示例。

empty_prob = torch.zeros(1, 1, 3, 3)  # 计算并保存当前步骤的中间状态。
empty_true = torch.zeros_like(empty_prob)  # 计算并保存当前步骤的中间状态。
false_positive = empty_prob.clone(); false_positive[..., 1, 1] = 1.0  # 计算并保存当前步骤的中间状态。
assert binary_overlap_metrics(empty_prob, empty_true)["iou"] == 1.0  # 用受控断言验证关键不变量。
assert binary_overlap_metrics(empty_prob, empty_true, empty_policy="skip")["evaluated"] == 0  # 用受控断言验证关键不变量。
assert binary_overlap_metrics(false_positive, empty_true)["dice"] == 0.0  # 用受控断言验证关键不变量。


## 7. 合成几何分割数据与独立随机流

每张图随机放置圆或矩形，mask 是精确真值；部分样本故意为空。输入由前景亮度、背景梯度和噪声组成。训练、validation、test 使用不同 seed，验证集只选阈值，测试集只在阈值冻结后报告。

这类前景和背景几乎线性可分，比真实边界、遮挡、弱标注和域偏移简单得多。它适合单元测试，不适合证明模型“会分割真实目标”。


In [ ]:
def make_segmentation_data(count, height, width, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    yy, xx = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")  # 计算并保存当前步骤的中间状态。
    images, masks = [], []  # 计算并保存当前步骤的中间状态。
    for index in range(count):  # 遍历输入元素以累积或检查结果。
        mask = torch.zeros(height, width, dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        if index % 7 != 0:  # 按当前条件选择后续控制路径。
            center_y = int(torch.randint(8, height - 7, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
            center_x = int(torch.randint(8, width - 7, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
            if index % 2 == 0:  # 按当前条件选择后续控制路径。
                radius = int(torch.randint(4, 7, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
                mask[((yy - center_y) ** 2 + (xx - center_x) ** 2) <= radius ** 2] = 1  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                half_h = int(torch.randint(3, 7, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
                half_w = int(torch.randint(3, 7, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
                mask[max(0, center_y-half_h):min(height, center_y+half_h),  # 执行当前语句以推进本节示例。
                     max(0, center_x-half_w):min(width, center_x+half_w)] = 1  # 计算并保存当前步骤的中间状态。
        gradient = torch.linspace(0, 0.12, width).repeat(height, 1)  # 计算并保存当前步骤的中间状态。
        noise = 0.05 * torch.randn((height, width), generator=generator)  # 计算并保存当前步骤的中间状态。
        image = (0.12 + gradient + 0.72 * mask + noise).clamp(0, 1)  # 计算并保存当前步骤的中间状态。
        images.append(image.unsqueeze(0)); masks.append(mask.unsqueeze(0))  # 执行当前语句以推进本节示例。
    return torch.stack(images), torch.stack(masks)  # 返回当前分支计算出的结果。

train_x23, train_y23 = make_segmentation_data(21, 31, 35, SEED + 1)  # 计算并保存当前步骤的中间状态。
val_x23, val_y23 = make_segmentation_data(7, 31, 35, SEED + 2)  # 计算并保存当前步骤的中间状态。
test_x23, test_y23 = make_segmentation_data(7, 31, 35, SEED + 3)  # 计算并保存当前步骤的中间状态。
assert train_x23.shape == train_y23.shape == (21, 1, 31, 35)  # 用受控断言验证关键不变量。
assert int((train_y23.flatten(1).sum(1) == 0).sum()) == 3  # 用受控断言验证关键不变量。
assert not torch.equal(train_x23[:7], val_x23)  # 用受控断言验证关键不变量。
assert set(torch.unique(train_y23).tolist()) == {0.0, 1.0}  # 用受控断言验证关键不变量。


## 8. 小样本过拟合：检查 forward、loss 与 optimizer

先固定初始化，再让网络反复看 21 张图。Adam 是为了在短 notebook 中快速验证链路；真实训练应在训练集内比较优化器、学习率、权重衰减和 scheduler，并记录像素采样与增强。

过拟合验收看 loss 显著下降、梯度有限非零、训练 Dice 很高。validation 结果不作为“泛化证明”。


In [ ]:
torch.manual_seed(SEED + 10)  # 执行当前语句以推进本节示例。
model23 = UNet(base=4).to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer23 = torch.optim.Adam(model23.parameters(), lr=0.015)  # 计算并保存当前步骤的中间状态。

model23.train()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss23 = float(segmentation_loss(model23(train_x23), train_y23))  # 计算并保存当前步骤的中间状态。
loss_curve23 = []  # 计算并保存当前步骤的中间状态。
for step in range(100):  # 遍历输入元素以累积或检查结果。
    optimizer23.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits23 = model23(train_x23)  # 计算并保存当前步骤的中间状态。
    loss23 = segmentation_loss(logits23, train_y23)  # 计算并保存当前步骤的中间状态。
    loss23.backward()  # 执行当前语句以推进本节示例。
    optimizer23.step()  # 执行当前语句以推进本节示例。
    loss_curve23.append(float(loss23.detach()))  # 执行当前语句以推进本节示例。

model23.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    train_prob23 = torch.sigmoid(model23(train_x23))  # 计算并保存当前步骤的中间状态。
    final_loss23 = float(segmentation_loss(model23(train_x23), train_y23))  # 计算并保存当前步骤的中间状态。
train_metrics23 = binary_overlap_metrics(train_prob23, train_y23, threshold=0.5)  # 计算并保存当前步骤的中间状态。

assert final_loss23 < initial_loss23 * 0.25  # 用受控断言验证关键不变量。
assert train_metrics23["dice"] > 0.95  # 用受控断言验证关键不变量。
assert len(loss_curve23) == 100  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in loss_curve23)  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss23, 4), "final_loss": round(final_loss23, 4),  # 执行当前语句以推进本节示例。
       "train": train_metrics23})  # 执行当前语句以推进本节示例。


## 9. 阈值只在 validation 选择

sigmoid 概率的默认阈值 0.5 并非永远最优，特别是在类别不平衡、代价不对称或概率未校准时。下面只用 validation 的 mean Dice 从预先声明的候选集选择阈值，随后冻结并一次性应用到 test。若多个候选分数完全相同，本示例固定选择较小阈值；具体业务也可以预先约定更偏 precision 的规则，但不能看过 test 再改变。

真实项目应同时报告 PR 曲线、不同对象尺寸、空/非空、设备/地区/时间分群，并避免反复查看 test 后修改候选阈值。


In [ ]:
def select_threshold(validation_scores, candidates):  # 定义本节可复用的核心函数。
    if not candidates or set(validation_scores) != set(candidates):  # 按当前条件选择后续控制路径。
        raise ValueError("候选阈值与验证分数字典必须非空且一一对应")  # 遇到非法合同立即显式失败。
    if not all(math.isfinite(validation_scores[threshold]) for threshold in candidates):  # 按当前条件选择后续控制路径。
        raise ValueError("验证分数必须为有限数")  # 遇到非法合同立即显式失败。
    return max(candidates, key=lambda threshold: (validation_scores[threshold], -threshold))  # 返回当前分支计算出的结果。

# 确定性 oracle：最高分优先；完全同分时稳定选择较小阈值。
oracle_candidates23 = [0.25, 0.50, 0.75]  # 计算并保存当前步骤的中间状态。
assert select_threshold({0.25: 0.7, 0.50: 0.9, 0.75: 0.8}, oracle_candidates23) == 0.50  # 用受控断言验证关键不变量。
assert select_threshold({0.25: 0.9, 0.50: 0.9, 0.75: 0.8}, oracle_candidates23) == 0.25  # 用受控断言验证关键不变量。

model23.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    val_prob23 = torch.sigmoid(model23(val_x23))  # 计算并保存当前步骤的中间状态。

threshold_candidates23 = [0.25, 0.35, 0.45, 0.50, 0.60, 0.70, 0.80]  # 计算并保存当前步骤的中间状态。
validation_scores23 = {  # 计算并保存当前步骤的中间状态。
    threshold: binary_overlap_metrics(val_prob23, val_y23, threshold=threshold)["dice"]  # 计算并保存当前步骤的中间状态。
    for threshold in threshold_candidates23  # 遍历输入元素以累积或检查结果。
}  # 执行当前语句以推进本节示例。
selected_threshold23 = select_threshold(validation_scores23, threshold_candidates23)  # 计算并保存当前步骤的中间状态。
# 阈值冻结之后才计算 test 预测和指标；test_y23 从未参与上面的选择。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    test_prob23 = torch.sigmoid(model23(test_x23))  # 计算并保存当前步骤的中间状态。
test_metrics23 = binary_overlap_metrics(test_prob23, test_y23,  # 计算并保存当前步骤的中间状态。
                                        threshold=selected_threshold23)  # 计算并保存当前步骤的中间状态。

assert selected_threshold23 == select_threshold(validation_scores23, threshold_candidates23)  # 用受控断言验证关键不变量。
assert all(math.isfinite(score) for score in validation_scores23.values())  # 用受控断言验证关键不变量。
assert 0.0 <= test_metrics23["iou"] <= 1.0  # 用受控断言验证关键不变量。
assert 0.0 <= test_metrics23["dice"] <= 1.0  # 用受控断言验证关键不变量。
assert test_metrics23["evaluated"] == len(test_y23)  # 用受控断言验证关键不变量。
print({"validation": validation_scores23, "selected": selected_threshold23,  # 执行当前语句以推进本节示例。
       "test": test_metrics23})  # 执行当前语句以推进本节示例。


## 10. 失败反例：未对齐就 concat，以及概率/logit 混用

奇数尺寸在第二次上采样后少一行一列，直接 `torch.cat` 会报 shape 错误。另一个常见静默错误是先 sigmoid，再把概率传给 `binary_cross_entropy_with_logits`：该函数会把概率再次当 logit 做 sigmoid，损失和梯度都被改变。

若概率阈值为 $t$，等价 logit 阈值是 $\log(t/(1-t))$；只有概率阈值 0.5 对应 logit 阈值 0。


In [ ]:
decoder_bad = torch.zeros(1, 4, 30, 34)  # 计算并保存当前步骤的中间状态。
skip_odd = torch.zeros(1, 4, 31, 35)  # 计算并保存当前步骤的中间状态。
concat_error23 = None  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    _ = torch.cat([decoder_bad, skip_odd], dim=1)  # 计算并保存当前步骤的中间状态。
except RuntimeError as exc:  # 捕获预期异常并验证失败分支。
    concat_error23 = str(exc)  # 计算并保存当前步骤的中间状态。

sample_logit = torch.tensor([2.0], requires_grad=True)  # 计算并保存当前步骤的中间状态。
sample_target = torch.tensor([1.0])  # 计算并保存当前步骤的中间状态。
correct_bce = F.binary_cross_entropy_with_logits(sample_logit, sample_target)  # 计算并保存当前步骤的中间状态。
wrong_double_sigmoid = F.binary_cross_entropy_with_logits(torch.sigmoid(sample_logit), sample_target)  # 计算并保存当前步骤的中间状态。
equivalent_logit_threshold = math.log(selected_threshold23 / (1 - selected_threshold23))  # 计算并保存当前步骤的中间状态。

assert concat_error23 is not None  # 用受控断言验证关键不变量。
assert align_to(decoder_bad, skip_odd).shape[-2:] == (31, 35)  # 用受控断言验证关键不变量。
assert not torch.isclose(correct_bce, wrong_double_sigmoid)  # 用受控断言验证关键不变量。
assert ((test_prob23 >= selected_threshold23) ==  # 用受控断言验证关键不变量。
        (torch.logit(test_prob23.clamp(1e-6, 1-1e-6)) >= equivalent_logit_threshold)).all()  # 计算并保存当前步骤的中间状态。
print("预期 concat 失败:", concat_error23.split("\n")[0])  # 执行当前语句以推进本节示例。


## 11. 全分辨率 tile：覆盖、halo 与边界效应

超大图通常不能一次放入显存。重叠 tiling 至少要保证：最后一个 tile 贴住右/下边界、每个像素覆盖次数大于零、重叠区用权重融合。直接平均 logits 是一种基线；对概率平均、中心裁剪或 Hann 权重会得到不同结果，必须版本化。

卷积依赖邻域，tile 边缘缺少整图上下文，因此即使重叠平均也不保证与整图推理完全相同。生产常增加 halo：读取更大的输入块，只保留中心可信区域；还要统一 padding mode、缩放尺度和坐标回写。


In [ ]:
def axis_starts(length, tile, overlap):  # 定义本节可复用的核心函数。
    if not (0 <= overlap < tile <= length):  # 按当前条件选择后续控制路径。
        raise ValueError("要求 0 <= overlap < tile <= length")  # 遇到非法合同立即显式失败。
    starts = list(range(0, length - tile + 1, tile - overlap))  # 计算并保存当前步骤的中间状态。
    if starts[-1] != length - tile:  # 按当前条件选择后续控制路径。
        starts.append(length - tile)  # 执行当前语句以推进本节示例。
    return starts  # 返回当前分支计算出的结果。

def tile_slices(height, width, tile=24, overlap=8):  # 定义本节可复用的核心函数。
    return [(slice(y, y + tile), slice(x, x + tile))  # 返回当前分支计算出的结果。
            for y in axis_starts(height, tile, overlap)  # 遍历输入元素以累积或检查结果。
            for x in axis_starts(width, tile, overlap)]  # 遍历输入元素以累积或检查结果。

full_image23, _ = make_segmentation_data(1, 47, 53, SEED + 20)  # 计算并保存当前步骤的中间状态。
accumulator = torch.zeros_like(full_image23)  # 计算并保存当前步骤的中间状态。
coverage = torch.zeros_like(full_image23)  # 计算并保存当前步骤的中间状态。
tiles23 = tile_slices(47, 53, tile=24, overlap=8)  # 计算并保存当前步骤的中间状态。
for ys, xs in tiles23:  # 遍历输入元素以累积或检查结果。
    accumulator[..., ys, xs] += full_image23[..., ys, xs]  # 计算并保存当前步骤的中间状态。
    coverage[..., ys, xs] += 1  # 计算并保存当前步骤的中间状态。
reconstructed23 = accumulator / coverage  # 计算并保存当前步骤的中间状态。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    full_logits23 = model23(full_image23)  # 计算并保存当前步骤的中间状态。
    tiled_logits23 = torch.zeros_like(full_logits23)  # 计算并保存当前步骤的中间状态。
    tile_weights23 = torch.zeros_like(full_logits23)  # 计算并保存当前步骤的中间状态。
    for ys, xs in tiles23:  # 遍历输入元素以累积或检查结果。
        tiled_logits23[..., ys, xs] += model23(full_image23[..., ys, xs])  # 计算并保存当前步骤的中间状态。
        tile_weights23[..., ys, xs] += 1  # 计算并保存当前步骤的中间状态。
    tiled_logits23 /= tile_weights23  # 计算并保存当前步骤的中间状态。
seam_gap23 = float((full_logits23 - tiled_logits23).abs().mean())  # 计算并保存当前步骤的中间状态。

assert float(coverage.min()) >= 1  # 用受控断言验证关键不变量。
assert torch.allclose(reconstructed23, full_image23)  # 用受控断言验证关键不变量。
assert tiles23[-1][0].stop == 47 and tiles23[-1][1].stop == 53  # 用受控断言验证关键不变量。
assert torch.isfinite(tiled_logits23).all()  # 用受控断言验证关键不变量。
print({"tiles": len(tiles23), "max_coverage": int(coverage.max()),  # 执行当前语句以推进本节示例。
       "whole_vs_tiled_mean_abs_gap": round(seam_gap23, 6)})  # 执行当前语句以推进本节示例。


## 12. 梯度、模型指纹与发布清单

分割制品必须绑定输入通道/色彩空间、归一化、输出类别次序、阈值、空 mask policy、resize/tiling 版本和权重。只保存 `.pt` 权重会让服务端无法判断 0.5 是 probability 阈值还是 logit 阈值。

下面再做一次反向传播，验证编码器、解码器和输出头都有有限非零梯度；随后对排序后的 `state_dict` 计算 SHA-256。


In [ ]:
model23.train()  # 执行当前语句以推进本节示例。
optimizer23.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
probe_loss23 = segmentation_loss(model23(train_x23[:4]), train_y23[:4])  # 计算并保存当前步骤的中间状态。
probe_loss23.backward()  # 执行当前语句以推进本节示例。
gradients23 = {name: float(parameter.grad.norm()) for name, parameter in model23.named_parameters()  # 计算并保存当前步骤的中间状态。
               if parameter.grad is not None}  # 按当前条件选择后续控制路径。

def state_fingerprint23(model):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(model.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(name.encode("utf-8"))  # 执行当前语句以推进本节示例。
        digest.update(str(value.dtype).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(str(tuple(value.shape)).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

fingerprint23 = state_fingerprint23(model23)  # 计算并保存当前步骤的中间状态。
manifest23 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "unet-base4-binary-toy-v1",  # 执行当前语句以推进本节示例。
    "architecture": {"base": 4, "down_levels": 2, "output_channels": 1},  # 执行当前语句以推进本节示例。
    "input": {"layout": "NCHW", "channels": 1, "range": [0.0, 1.0]},  # 执行当前语句以推进本节示例。
    "output": {"kind": "pixel_logits", "threshold_space": "probability",  # 执行当前语句以推进本节示例。
               "threshold": selected_threshold23, "empty_policy": "perfect"},  # 执行当前语句以推进本节示例。
    "tiling": {"tile": 24, "overlap": 8, "fusion": "uniform_logit_average_demo"},  # 执行当前语句以推进本节示例。
    "seed": SEED, "torch": torch.__version__, "state_dict_sha256": fingerprint23,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

assert gradients23["inc.net.0.weight"] > 0  # 用受控断言验证关键不变量。
assert gradients23["up1.up.weight"] > 0  # 用受控断言验证关键不变量。
assert gradients23["out_conv.weight"] > 0  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in gradients23.values())  # 用受控断言验证关键不变量。
assert len(fingerprint23) == 64  # 用受控断言验证关键不变量。
assert state_fingerprint23(model23) == fingerprint23  # 用受控断言验证关键不变量。
print(json.dumps(manifest23, ensure_ascii=False, indent=2))  # 计算并保存当前步骤的中间状态。


## 13. 生产边界与资料

真实分割还要解决：按患者/地块/视频来源切分防泄漏；标注版本和 annotator 一致性；类别长尾与小目标采样；多尺度增强但保持 mask 插值为 nearest；连通域/孔洞等后处理；边界指标 Hausdorff/Boundary IoU；推理显存和 tile halo；跨设备/地域漂移；阈值校准、人工复核与回滚。

若任务是多类互斥分割，应改为 `[N,C,H,W]` logits + cross-entropy，并用 argmax；若是多标签像素任务，则每通道独立 sigmoid 和阈值。两者不可仅靠改 `out_channels` 混用。

原始论文与官方资料：

- Ronneberger、Fischer、Brox，[U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)，2015。
- Milletari 等，[V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation](https://arxiv.org/abs/1606.04797)，Dice loss 的相关来源。
- PyTorch 官方文档：[BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)、[ConvTranspose2d](https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)、[`interpolate`](https://pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html)。

结论边界：这里只证明手写 U-Net 在受控奇数尺寸合成图上能执行、反传并过拟合；没有证明真实分割质量、跨域鲁棒性或临床/安全可用性。
